In [ ]:
#!/usr/bin/env python3

import os
import json
import argparse
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from huggingface_hub import hf_hub_download


def load_model_and_tokenizer(path_or_repo):
    """
    Load base model + PEFT adapter + tokenizer.
    Tries local folder first, otherwise downloads from Hugging Face Hub.
    """
    if os.path.exists(path_or_repo):
        print(f"Loading model from local folder: {path_or_repo}")
        adapter_path = os.path.join(path_or_repo, "adapter_config.json")
        chat_template_path = os.path.join(path_or_repo, "chat_template.jinja")
        if not os.path.exists(chat_template_path):
            chat_template_path = os.path.join("chat_template.jinja")
    else:
        print(f"Local folder not found. Downloading from Hugging Face repo: {path_or_repo}")
        adapter_path = hf_hub_download(repo_id=path_or_repo, filename="adapter_config.json")
        chat_template_path = hf_hub_download(repo_id=path_or_repo, filename="chat_template.jinja")

    adapter_config = json.load(open(adapter_path))
    base_model_name_or_path = adapter_config["base_model_name_or_path"]
    chat_template = open(chat_template_path).read()

    tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path)
    tokenizer.chat_template = chat_template
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(base_model_name_or_path)
    model.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(model, path_or_repo, is_trainable=False)
    model.eval()
    return model, tokenizer




In [ ]:

model_path = "./models/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42"
model_name = "fdsg"
save_dir = "/tmp/"
os.makedirs(save_dir, exist_ok=True)

print("Loading dataset...")
dataset = load_dataset("loris3/ftrace_date-lm_conversational")["train"]

dataset = dataset.select(range(10))

# Load model and tokenizer
model, tokenizer = load_model_and_tokenizer(model_path)
device = 0 if torch.cuda.is_available() else -1
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

prediction_file = os.path.join(save_dir, "task-000-ftrace_date-lm_conversational-predictions.jsonl")
correct_list = []


Loading dataset...
Loading model from local folder: ./models/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (6 outputs left out)

Device set to use cuda:0


Running evaluation...


100%|██████████| 10/10 [00:00<00:00, 11.78it/s]


Exact-Match Accuracy: 0.6000
Results saved to /tmp/


In [ ]:
def get_exact_match(examples):
    print(examples)

    user_prompts = [m[0]["content"] for m in examples["messages"]]
    references = [m[1]["content"].strip() for m in examples["messages"]]

    # Generate model response
    output = generator(
        user_prompts,
        max_new_tokens=200,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        return_full_text=False
    )
    print(output)
    generated = {
    "generated_texts": [item[0]['generated_text'].strip() for item in output]
    }
    exact_match = generated == reference

    result = {}
    # {
    #     "id": example["id"],
    #     "prompt": user_prompt,
    #     "reference": reference,
    #     "generated": generated,
    #     "exact_match": exact_match
    # }
    result.extend(user_prompts)
    print(result)
    return result

dataset.map(get_exact_match, batched=True, batch_size=4)



Map:   0%|          | 0/10 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Algeria


ValueError: I/O operation on closed file.